In [2]:
from typing import List, Dict, Tuple
from collections import defaultdict
import numpy as np


def count_ngrams(sentences: List[List[str]], n: int, start_token: str = '<s>', end_token: str = '<e>') -> Dict[Tuple, int]:
    """Count n-grams from sentences."""
    ngram_counts = defaultdict(int)

    for sent in sentences:
        padded = [start_token] * (n - 1) + sent + [end_token]

        for i in range(len(padded) - n + 1):
            ngram = tuple(padded[i:i+n])
            ngram_counts[ngram] += 1

    return dict(ngram_counts)


def estimate_probability(word: str, prev_ngram: Tuple, ngram_counts: Dict,
                         prev_ngram_counts: Dict, vocab_size: int, k: float = 1.0) -> float:
    """Estimate probability of word given previous n-gram with k-smoothing."""
    full_ngram = prev_ngram + (word,)

    numerator = ngram_counts.get(full_ngram, 0) + k
    denominator = prev_ngram_counts.get(prev_ngram, 0) + k * vocab_size

    return numerator / denominator if denominator > 0 else 0


def estimate_all_probabilities(prev_ngram: Tuple, vocabulary: set, ngram_counts: Dict,
                               prev_ngram_counts: Dict, vocab_size: int, k: float = 1.0) -> Dict[str, float]:
    """Estimate probabilities for all words given previous n-gram."""
    probs = {}
    for word in vocabulary:
        probs[word] = estimate_probability(word, prev_ngram, ngram_counts,
                                          prev_ngram_counts, vocab_size, k)
    return probs


def make_count_matrix(ngram_counts: Dict, vocabulary: set, n: int) -> Dict:
    """Create count matrix for (n+1)-grams."""
    matrix = {}
    for ngram, count in ngram_counts.items():
        if len(ngram) == n:
            prev = ngram[:-1]
            matrix[prev] = matrix.get(prev, {})
            matrix[prev][ngram[-1]] = count
    return matrix


def make_probability_matrix(ngram_counts: Dict, prev_ngram_counts: Dict,
                           vocabulary: set, k: float = 1.0) -> Dict:
    """Create probability matrix for n-grams."""
    vocab_size = len(vocabulary)
    matrix = {}

    for prev_ngram in prev_ngram_counts.keys():
        matrix[prev_ngram] = estimate_all_probabilities(
            prev_ngram, vocabulary, ngram_counts, prev_ngram_counts, vocab_size, k
        )

    return matrix


class LanguageModel:
    def __init__(self, vocabulary: set, k: float = 1.0):
        self.vocabulary = vocabulary
        self.k = k
        self.vocab_size = len(vocabulary)
        self.ngram_counts = {}
        self.prob_matrix = {}

    def train(self, sentences: List[List[str]], n: int):
        """Train n-gram model."""
        self.ngram_counts = count_ngrams(sentences, n)

        prev_grams = defaultdict(int)
        for ngram in self.ngram_counts.keys():
            prev = ngram[:-1]
            prev_grams[prev] += self.ngram_counts[ngram]

        self.prob_matrix = make_probability_matrix(
            self.ngram_counts, dict(prev_grams), self.vocabulary, self.k
        )

        return self

    def get_probability(self, prev_ngram: Tuple, word: str) -> float:
        """Get probability of word given previous n-gram."""
        if prev_ngram in self.prob_matrix:
            return self.prob_matrix[prev_ngram].get(word, self.k / (prev_ngram_counts.get(prev_ngram, 0) + self.k * self.vocab_size))
        return self.k / (self.k * self.vocab_size)